<p style='margin-top: 1rem; margin-bottom: 1rem;'>Developed by Marco Frodl, Principal Consultant for Generative AI @ <a href='https://go.mfr.one/tt-en' _target='blank'>Thinktecture AG</a> -- More about me on my <a href='https://go.mfr.one/marcofrodl-en' _target='blank'>profile page</a></p>
<p style='margin-top: 1rem; margin-bottom: 1rem;'>Revised for LangChain 1.0+ by Lukas Köhler, Generative AI Developer @ <a href='https://thinktecture.com' _target='blank'>Thinktecture AG</a> -- More about me on my <a href='https://www.thinktecture.com/thinktects/lukas-koehler/' _target='blank'>profile page</a></p>

We want to implement tool calling using LangChain 1.0 and Mistral Small LLM.

Tasks and details:

* All parts where you should implement something are marked with "..." or a sentence starting with "Task: "
* Include tracing with Langfuse if you have a key for it. If not, remove all the callback parts from the invoke instructions
* Use Mistral-Small-3.2-24B-Instruct-2506 via OVH API Endpoints


If you need help with LangChain, use the LangChain chat bot: https://chat.langchain.com/

## API keys

In [ ]:
# In Codespaces the keys come from the Codespaces secrets, locally from a .env file in this folder.
# Langfuse reads LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY directly from the environment.
import os

from dotenv import load_dotenv

load_dotenv()

for name in ("OVH_API_KEY", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"):
    print(name, "✅" if os.environ.get(name) else "❌ missing")

## Imports

In [ ]:
# Python
import requests
from pprint import pprint
from typing import Literal
from IPython.display import Image, display
# LangChain
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent # NEW in LangChain 1.0+
# LangFuse
from langfuse.langchain import CallbackHandler

## Prepare LangFuse

In [ ]:
# prepare Langfuse as debugging and tracing framework for our Generative AI application - never develop GenAI apps without that!
handler = CallbackHandler()

## Prepare LLM

In [ ]:
# for this notebook we use the OVH AI endpoints and get access to a bunch of different open source LLMs
from langchain_openai import ChatOpenAI

TEMPERATURE = 0
MAX_TOKENS = 1500

# OVH Mistral Small 3.2 24B
MODEL_NAME_MISTRAL = "..."
BASE_URL_MISTRAL = "https://oai.endpoints.kepler.ai.cloud.ovh.net/v1"

llm = ChatOpenAI(
    model=MODEL_NAME_MISTRAL,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    base_url=BASE_URL_MISTRAL,
    api_key=os.environ["OVH_API_KEY"],
)

## Define Tools

In [ ]:
def get_project_details_by_project_symbol(project_symbol: str) -> str:
  """
  Use this tool when the user asks about project details or related information (e.g., project manager) and a project symbol is mentioned.
  Extract the project symbol from the user's query, which is typically the name of the organization or company related to a project, initiative, or undertaking.
  ## Instructions:
  * Focus on proper nouns indicating entities like companies or institutions.
  * Handle queries in multiple languages (e.g., English, German, French).
  * Ignore unrelated words or phrases surrounding the project symbol.
  * If no project symbol is found, use "None" as project_symbol input.

  Returns a dictionary with project_symbol, project_id and project_name
  For unknown project symbols returns {"project_symbol": None, "project_id": None, "project_name": None}
  """
  print("---TOOL get_project_details_by_project_symbol---")
  project_mapping = {
      'post': ("POST","tt-pt-25b", "Deutsche Post AG - AI Assistent for parcel tracking"),
      'siemens': ("SIEMENS","tt-si-24f", "Siemens AG - Wind Turbine Optimization"),
      'mercedes': ("MERCEDES","tt-me-24a", "Mercedes-Benz AG - Electric Vehicle Charging Station"),
  }

  p_symbol, p_id, p_name = project_mapping.get(project_symbol.lower(), (None, None, None))
  answer = f"project_symbol: {p_symbol}, project_id: {p_id}, project_name: {p_name}"
  print(answer)
  return answer

def get_manager_details_by_project_id(project_id: str) -> str:
  """
    Use this tool if the user requests details about a project manager for a given project_id. Project_id always starts with 'tt-'.
    Given a user query in any language, determine whether the user is requesting details about project_manager associated with a specific project_id.
    If such a request is identified, extract or infer the intent and proceed to look up the project manager’s details using the provided project_id

    ## Instructions:
    * Recognize indirect references to the project manager, such as:
      * "contact person"
      * "responsible individual"
      * "who is in charge"
      * "Ansprechpartner" (German)
      * "responsable du projet" (French)
    * Do **not** rely solely on the phrase "project manager" — interpret the user’s intent.
    * Only respond if the request **clearly asks** for contact or identity details of the person managing or responsible for the project.
    * Ignore general project_id queries that don't mention or imply interest in the person responsible.

    ## Examples:
    * **Input**: "Can you give me the project manager details for project_id tt-123-8b?" - **Output**: Manager details: {"first_name": "Max", "last_name": "Mustermann", "gender": "Herr", "email": "max.mustermann@muster-ag.de", "telephone": "089-4577834"}
    * **Input**: "Wer ist der Ansprechpartner für project_id tt-123-8b?" - **Output**: Manager details: {"first_name": "Max", "last_name": "Mustermann", "gender": "Herr", "email": "max.mustermann@muster-ag.de", "telephone": "089-4577834"}
    * **Input**: "Ich brauche die Kontaktdaten der zuständigen Person für project_id tt-123-8b." - **Output**: Manager details: {"first_name": "Max", "last_name": "Mustermann", "gender": "Herr", "email": "max.mustermann@muster-ag.de", "telephone": "089-4577834"}

    Returns dict project_manager details for given project_id not None, empty dict otherwise.
  """
  print("---TOOL get_manager_details_by_project_id---")
  details = {
      "tt-pt-25b": {"first_name": "Alice", "last_name": "Mueller", "gender": "Frau",
          "email": "alice.mueller@post-ag.com", "telephone": "0123456789"},
      "tt-si-24f": {"first_name": "Bob", "last_name": "Schmidt", "gender": "Herr",
          "email": "bob.schmidt@siemens.com", "telephone": "0987654321"},
      "tt-me-24a": {"first_name": "Clara", "last_name": "Fischer", "gender": "Frau",
          "email": "clara.fischer@mercedes-ev.com", "telephone": "1234567890"}
  }
  manager_details = details.get(project_id.lower(), {})
  answer = f"Manager details: {manager_details}"
  print(answer)
  return answer

In [ ]:
# Task: Test the tools
# What are the project details for the project Post?
print("Test call result: " + get_project_details_by_project_symbol(project_symbol="..."))

# Who is the project manager for the project Post?
print("Test call result: " + get_manager_details_by_project_id(project_id="..."))

# Create Agent with LangChain 1.0+ create_agent()

In [ ]:
# Task: Define the agent using create_agent(), the definition can be found here: https://docs.langchain.com/oss/python/langchain/overview

agent = ...

## Test Drive

In [ ]:
message1 = [HumanMessage(content="An welchem Tag beginnt der Sommer?")]
message2 = [HumanMessage(content="Projektname für Vorhaben Siemens")]
message3 = [HumanMessage(content="Wer leitet das Post-Projekt?")]

In [ ]:
result = agent.invoke({"messages": message1}, config={"callbacks": [handler]})

# output everything
pprint(result, sort_dicts=False)
print("-" * 10)

# Get the content of the final AIMessage
final_response = result["messages"][-1].content

# Output the content
print(final_response)

In [ ]:
result = agent.invoke({"messages": message2}, config={"callbacks": [handler]})

# output everything
pprint(result, sort_dicts=False)
print("-" * 10)

# Get the content of the final AIMessage
final_response = result["messages"][-1].content

# Output the content
print(final_response)

In [ ]:
result = agent.invoke({"messages": message3}, config={"callbacks": [handler]})

# output everything
pprint(result, sort_dicts=False)
print("-" * 10)

# Get the content of the final AIMessage
final_response = result["messages"][-1].content

# Output the content
print(final_response)